# 05 - Benchmark Comparison

Compare adapted methods against the realized VoV benchmark.


In [1]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

df = pd.read_csv("data_vov_methods.csv", parse_dates=["date_ad"], index_col="date_ad")
benchmark = "benchmark_vov_30d"
methods = ["stvv_historical", "emvv_adapted", "mivv_adapted_garch", "mivv_adapted_egarch"]
eval_df = df[[benchmark] + methods].dropna()
len(eval_df), eval_df.head()


(2536,
             benchmark_vov_30d  stvv_historical  emvv_adapted  \
 date_ad                                                        
 2015-04-05           0.016857         0.149182      0.004350   
 2015-04-06           0.016214         0.149479      0.004361   
 2015-04-07           0.015117         0.117279      0.004296   
 2015-04-08           0.013823         0.116691      0.004363   
 2015-04-09           0.012376         0.115381      0.004389   
 
             mivv_adapted_garch  mivv_adapted_egarch  
 date_ad                                              
 2015-04-05            0.003294             0.006638  
 2015-04-06            0.003305             0.006696  
 2015-04-07            0.003327             0.006686  
 2015-04-08            0.002381             0.005272  
 2015-04-09            0.002333             0.005263  )

In [2]:
def metrics(y, yhat):
    err = yhat - y
    eps = 1e-12
    return {
        "MAE": np.mean(np.abs(err)),
        "MSE": np.mean(err ** 2),
        "MAPE": np.mean(np.abs(err) / np.maximum(np.abs(y), eps)) * 100,
        "WMAPE": np.sum(np.abs(err)) / np.maximum(np.sum(np.abs(y)), eps) * 100,
        "Spearman": spearmanr(y, yhat).statistic,
    }

comparison = pd.DataFrame(
    {method: metrics(eval_df[benchmark], eval_df[method]) for method in methods}
).T
comparison.to_csv("table_method_comparison.csv")
comparison


,MAE,MSE,MAPE,WMAPE,Spearman
stvv_historical,13.715962,713.007456,50358.107488,30377.248217,0.517561
emvv_adapted,0.042382,0.006161,79.753871,93.865787,0.521633
mivv_adapted_garch,0.035345,0.005043,102.925650,78.278841,0.413900
mivv_adapted_egarch,0.035313,0.005008,125.933312,78.209833,0.407741


In [3]:
regime_df = eval_df.copy()
regime_df["vol_regime"] = pd.qcut(regime_df[benchmark], q=3, labels=["low", "medium", "high"])

regime_tables = []
for regime, group in regime_df.groupby("vol_regime", observed=True):
    table = pd.DataFrame(
        {method: metrics(group[benchmark], group[method]) for method in methods}
    ).T
    table.insert(0, "regime", regime)
    regime_tables.append(table)

regime_comparison = pd.concat(regime_tables)
regime_comparison.to_csv("table_method_comparison_by_regime.csv")
regime_comparison


,regime,MAE,MSE,MAPE,WMAPE,Spearman
stvv_historical,low,3.451945,43.849156,69863.397460,71244.139110,0.553969
emvv_adapted,low,0.003149,0.000016,58.307927,64.990059,0.436869
mivv_adapted_garch,low,0.006024,0.000062,186.403999,124.319534,0.458671
mivv_adapted_egarch,low,0.007526,0.000079,262.713343,155.329151,0.366744
stvv_historical,medium,9.251039,208.667832,51207.080665,44607.126082,0.033439
emvv_adapted,medium,0.018200,0.000403,85.908557,87.758891,0.124163
mivv_adapted_garch,medium,0.010281,0.000167,47.810879,49.575756,-0.063070
mivv_adapted_egarch,medium,0.009043,0.000135,41.153661,43.606392,-0.000801
stvv_historical,high,28.427492,1885.118266,30027.903753,25891.284223,0.215300
emvv_adapted,high,0.105723,0.018050,95.027054,96.290702,0.284209
